# Shard merge: Combine all shards into one dataset

**What this does:** takes all `shard_<NAME>.pt` files produced by `shard_collect.ipynb` and concatenates them into one `(X, y, problem_ids)` dataset for the probe trainer.

**Runs on CPU. No GPU needed, no compute units consumed**

## What this saves

- `X.pt` -> merged hidden states, shape `[total_positions, HIDDEN_DIM]`
- `y.pt` -> merged labels, shape `[total_positions]`
- `problem_ids.pt` -> which problem each position came from, shape `[total_positions]`
- `merge_meta.json` -> metadata recording which shards were merged and consistency checks

In [14]:
# === Colab bootstrap ===
# import os

# IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

# if IN_COLAB:
#     from google.colab import drive
#     drive.mount("/content/drive")
#     OUTPUT_DIR = "/content/drive/MyDrive/cs639-outputs"
# else:
#     OUTPUT_DIR = "."

# os.makedirs(OUTPUT_DIR, exist_ok=True)
# print(f"IN_COLAB={IN_COLAB}  OUTPUT_DIR={OUTPUT_DIR}")

import os
OUTPUT_DIR = "./outputs"

In [15]:
%pip install -q torch pandas h5py


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip3.14 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Config

In [16]:
# Folder that holds all shard_*.pt files. Default is the same OUTPUT_DIR.
# If teammates sent you their shards separately, put them all in one folder
# and point SHARDS_DIR at it.
SHARDS_DIR = OUTPUT_DIR

# Output paths for the merged dataset
X_PATH           = os.path.join(OUTPUT_DIR, "X.pt")
Y_PATH           = os.path.join(OUTPUT_DIR, "y.pt")
PID_PATH         = os.path.join(OUTPUT_DIR, "problem_ids.pt")
MERGE_META_PATH  = os.path.join(OUTPUT_DIR, "merge_meta.json")

# Require at least this many shards; error otherwise (safety against merging
# with missing shards by accident).
MIN_EXPECTED_SHARDS = 1

In [17]:
# === Discover shard files ===
import glob

pattern = os.path.join(SHARDS_DIR, "shard_*.h5")
shard_paths = sorted(glob.glob(pattern))

print(f"Looking in: {SHARDS_DIR}")
print(f"Found {len(shard_paths)} shard file(s):")
for p in shard_paths:
    print(f"  {os.path.basename(p)}  ({os.path.getsize(p) / 1e6:.1f} MB)")

assert len(shard_paths) >= MIN_EXPECTED_SHARDS, (
    f"Only found {len(shard_paths)} shards; expected at least {MIN_EXPECTED_SHARDS}. "
    "Did all teammates upload their shards?"
)

Looking in: ./outputs
Found 6 shard file(s):
  shard_Dharini.h5  (143.9 MB)
  shard_RRR.h5  (97.5 MB)
  shard_avni.h5  (158.7 MB)
  shard_avni2.h5  (178.6 MB)
  shard_gayu_250_375.h5  (176.9 MB)
  shard_gayu_375_500.h5  (175.5 MB)


In [18]:
# === Load + inspect each shard ===
import h5py
import torch
import pandas as pd

shards_data = []  # (path, attrs_dict, set_of_pids)
rows = []

for p in shard_paths:
    with h5py.File(p, 'r') as f:
        attrs = dict(f.attrs)
        pids = {int(k.split('_', 1)[1]) for k in f.keys() if k.startswith('problem_')}
        n_pos     = sum(f[f"problem_{pid}"]["hidden_states"].shape[0] for pid in pids)
        n_correct = sum(int(f[f"problem_{pid}"]["label"][()]) for pid in pids)
    shards_data.append((p, attrs, pids))
    rows.append({
        "file":            os.path.basename(p),
        "NAME":            attrs.get("name", "?"),
        "range":           f"{attrs.get('start_idx', '?')}..{attrs.get('end_idx', '?')}",
        "positions":       n_pos,
        "unique_problems": len(pids),
        "correct":         n_correct,
        "incorrect":       n_pos - n_correct,
        "MAX_NEW_TOKENS":  attrs.get("max_new_tokens"),
        "PROBE_LAYER":     attrs.get("probe_layer"),
        "HIDDEN_DIM":      attrs.get("hidden_dim"),
        "MODEL_ID":        attrs.get("model_id"),
    })

summary = pd.DataFrame(rows)
summary

,file,NAME,range,positions,unique_problems,correct,incorrect,MAX_NEW_TOKENS,PROBE_LAYER,HIDDEN_DIM,MODEL_ID
0,shard_Dharini.h5,Dharini,500..602,9737,102,52,9685,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
1,shard_RRR.h5,RRR,650..720,6593,70,36,6557,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
2,shard_avni.h5,avni,0..120,10731,120,78,10653,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
3,shard_avni2.h5,avni2,120..250,12072,130,74,11998,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
4,shard_gayu_250_375.h5,gayu,250..375,11971,125,78,11893,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
5,shard_gayu_375_500.h5,gayu,375..500,11881,125,69,11812,8192,-1,3584,deepseek-ai/DeepSeek-R1-Distill-Qwen-7B


In [19]:
# === Consistency checks: shards must share the same protocol ===
def _unique(col):
    vals = [r[col] for r in rows if r[col] is not None]
    return set(vals)

for field in ["MAX_NEW_TOKENS", "PROBE_LAYER", "HIDDEN_DIM", "MODEL_ID"]:
    uniq = _unique(field)
    assert len(uniq) <= 1, (
        f"Inconsistent {field} across shards: {uniq}. "
        "Regenerate any mismatched shards with the shared protocol."
    )
    print(f"  {field}: {uniq.pop() if uniq else '(none recorded)'}")

# Check that no two shards cover the same problem_ids (overlap = double-counted data)
all_pids_per_shard = [pids for _, _, pids in shards_data]
for i in range(len(shards_data)):
    for j in range(i + 1, len(shards_data)):
        overlap = all_pids_per_shard[i] & all_pids_per_shard[j]
        if overlap:
            print(f"⚠️  OVERLAP between {os.path.basename(shard_paths[i])} and "
                  f"{os.path.basename(shard_paths[j])}: {len(overlap)} shared problems")
            print(f"    example overlapping problem_ids: {sorted(overlap)[:10]}")
print("Overlap check complete.")

  MAX_NEW_TOKENS: 8192
  PROBE_LAYER: -1
  HIDDEN_DIM: 3584
  MODEL_ID: deepseek-ai/DeepSeek-R1-Distill-Qwen-7B
Overlap check complete.


In [20]:
# === Concatenate ===
import h5py

X_parts, y_parts, pid_parts = [], [], []
for p, _, pids in shards_data:
    with h5py.File(p, 'r') as f:
        for pid in sorted(pids):
            grp = f[f"problem_{pid}"]
            hs = torch.from_numpy(grp["hidden_states"][...]).float()
            label = int(grp["label"][()])
            if hs.shape[0] == 0:
                continue
            X_parts.append(hs)
            y_parts.extend([label] * hs.shape[0])
            pid_parts.extend([pid] * hs.shape[0])

X = torch.cat(X_parts, dim=0)
y = torch.tensor(y_parts, dtype=torch.float32)
pids = torch.tensor(pid_parts, dtype=torch.long)

print(f"Merged X:           {tuple(X.shape)}  dtype={X.dtype}")
print(f"Merged y:           {tuple(y.shape)}  dtype={y.dtype}")
print(f"Merged problem_ids: {tuple(pids.shape)}  dtype={pids.dtype}")
print(f"Unique problems:    {len(torch.unique(pids))}")
print(f"Label distribution: {int(y.sum())} correct / {int((1 - y).sum())} incorrect "
      f"({y.mean().item():.1%} positive)")


Merged X:           (62985, 3584)  dtype=torch.float32
Merged y:           (62985,)  dtype=torch.float32
Merged problem_ids: (62985,)  dtype=torch.int64
Unique problems:    672
Label distribution: 34869 correct / 28116 incorrect (55.4% positive)


In [21]:
# === Save merged dataset + metadata ===
import json
import h5py
import numpy as np

def _json_default(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

# --- .pt tensors for probe_train.ipynb ---
torch.save(X, X_PATH)
torch.save(y, Y_PATH)
torch.save(pids, PID_PATH)

# --- merged HDF5 for downstream tasks ---
HDF5_PATH = os.path.join(OUTPUT_DIR, "traces_all.h5")
with h5py.File(HDF5_PATH, "w") as out:
    for p, _, shard_pids in shards_data:
        with h5py.File(p, "r") as src:
            for pid in sorted(shard_pids):
                src.copy(f"problem_{pid}", out)

with open(MERGE_META_PATH, "w") as f:
    json.dump({
        "shards": rows,
        "total_positions": int(X.shape[0]),
        "hidden_dim": int(X.shape[1]),
        "unique_problems": int(len(torch.unique(pids))),
        "n_correct_positions": int(y.sum()),
        "n_incorrect_positions": int((1 - y).sum()),
    }, f, indent=2, default=_json_default)

print(f"✓ X saved           -> {X_PATH}")
print(f"✓ y saved           -> {Y_PATH}")
print(f"✓ problem_ids saved -> {PID_PATH}")
print(f"✓ traces_all saved  -> {HDF5_PATH}")
print(f"✓ merge_meta saved  -> {MERGE_META_PATH}")


✓ X saved           -> ./outputs/X.pt
✓ y saved           -> ./outputs/y.pt
✓ problem_ids saved -> ./outputs/problem_ids.pt
✓ traces_all saved  -> ./outputs/traces_all.h5
✓ merge_meta saved  -> ./outputs/merge_meta.json


## Next step

- Open `probe_train.ipynb` (same folder) and run it.
- It will load `X.pt`, `y.pt`, `problem_ids.pt` from `OUTPUT_DIR`
- Train the linear probe with proper train/val/test splits and metrics (ROC-AUC, ECE, Brier). Runs on CPU in a few minutes, no compute units needed.